In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

In [2]:
warnings.filterwarnings("ignore")

In [3]:
diamonds = sns.load_dataset("diamonds")

In [5]:
diamonds.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [6]:
diamonds.shape

(53940, 10)

In [ ]:
diamonds.describe()

In [ ]:
diamonds.describe(exclude=np.number)

In [4]:
from sklearn.model_selection import train_test_split

In [9]:
X,y = diamonds.drop('price', axis = 1),diamonds[['price']]

In [ ]:
diamonds.dtypes

In [ ]:
cats = X.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:
cats


In [ ]:
for col in cats:
    X[col] = X[col].astype('category')

In [ ]:
X.dtypes

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=1)

In [27]:
import xgboost as xgb

In [ ]:
dtrain_reg = xgb.DMatrix(X_train, y_train, enable_categorical=True)

In [ ]:
dtest_reg = xgb.DMatrix(X_test, y_test, enable_categorical=True)

In [ ]:
params= {"objective": "reg:squarederror", "tree_method": "hist"}

In [ ]:
n=100
model = xgb.train(params= params, dtrain=dtrain_reg, num_boost_round =n,)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
preds = model.predict(dtest_reg)

In [ ]:
rmse = mean_squared_error(y_test, preds,squared=False)

In [ ]:
print(f"RMSE of the bae model : {rmse:.3f}")

In [ ]:
params= {"objective": "reg:squarederror", "tree_method": "hist"}

In [ ]:
n=10000

In [ ]:
evals =[(dtest_reg, "validation"), (dtrain_reg, "train")]

In [ ]:
#Early stopping - does not work in this setup

In [ ]:
model = xgb.train(params=params, dtrain=dtrain_reg, num_boost_round = n, evals = evals,verbose_eval = 250,early_stopping_rounds=50)

In [ ]:
# XGBoost Cross-Validation

In [ ]:
n = 1000

In [ ]:
results = xgb.cv(params,dtrain_reg, num_boost_round=n, nfold=5, early_stopping_rounds = 20)

In [ ]:
results.head()

In [ ]:
best_rmse = results['test-rmse-mean'].min()

In [ ]:
best_rmse

In [ ]:
#XGBoost Classification


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
X, y = diamonds.drop("cut", axis=1), diamonds[["cut"]]

In [ ]:
y_encoded = OrdinalEncoder().fit_transform(y)

In [ ]:
cats = X.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:
cats

In [ ]:
for col in cats:
    X[col] = X[col].astype('category')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y_encoded, random_state =2, stratify=y_encoded)

In [ ]:
dtrain_clf =xgb.DMatrix(X_train, y_train, enable_categorical=True)
dtest_clf = xgb.DMatrix(X_test, y_test, enable_categorical=True)

In [ ]:
params={"objective" : "multi:softprob", "treemethod" : "hist", "num_class" : 5}


In [ ]:
n = 1000

In [ ]:
results = xgb.cv(params, dtrain_clf, num_boost_round=n, nfold=5, metrics=["mlogloss", "auc", "merror"])

In [ ]:
results.keys()

In [ ]:
results['test-auc-mean'].max()

In [ ]:
#XGBoost using Sklearn. Categorical is not working in this version.

In [ ]:
xgb_classifier = xgb.XGBClassifier(n_estimators=100, objetive='binary:logistic',tree_method="gpu_hist",eta=0.1,max_depth=3)

In [12]:
X,y = diamonds.drop('cut', axis = 1),diamonds[['cut']]

In [19]:
y = pd.get_dummies(y,columns=['cut'])

In [20]:
y

,cut_Ideal,cut_Premium,cut_Very Good,cut_Good,cut_Fair
0,1,0,0,0,0
1,0,1,0,0,0
2,0,0,0,1,0
3,0,1,0,0,0
4,0,0,0,1,0
...,...,...,...,...,...
53935,1,0,0,0,0
53936,0,0,0,1,0
53937,0,0,1,0,0
53938,0,1,0,0,0


In [31]:
X_tr, X_te, y_tr, y_te = train_test_split(X,y,test_size=0.2) 

In [32]:
xgb_model = xgb.XGBClassifier()

In [33]:
xgb_model.fit(X_tr, y_tr)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, The experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:color: category, clarity: category